In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split

In [2]:
tracksPath = "Data/unzippedRaw/fma_metadata/tracks.csv"

tracks = pd.read_csv(tracksPath, index_col=0, header=[0, 1])

print(tracks.shape)
tracks.head()

(106574, 52)


album                                                     \
         comments         date_created        date_released engineer   
track_id                                                               
2               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
3               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
5               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
10              0  2008-11-26 01:45:08  2008-02-06 00:00:00      NaN   
20              0  2008-11-26 01:45:05  2009-01-06 00:00:00      NaN   

                                                                          \
         favorites id                                information listens   
track_id                                                                   
2                4  1                                    <p></p>    6073   
3                4  1                                    <p></p>    6073   
5                4  1                                    <p></p>    6073   
10               4  6                                        NaN   47632   
20               2  4  <p> "spiritual songs" from Nicky Cook</p>    2710   

                        ...       track                         \
         producer tags  ... information interest language_code   
track_id                ...                                      
2             NaN   []  ...         NaN     4656            en   
3             NaN   []  ...         NaN     1470            en   
5             NaN   []  ...         NaN     1933            en   
10            NaN   []  ...         NaN    54881            en   
20            NaN   []  ...         NaN      978            en   

                                                                              \
                                                    license listens lyricist   
track_id                                                                       
2         Attribution-NonCommercial-ShareAlike 3.0 Inter...    1293      NaN   
3         Attribution-NonCommercial-ShareAlike 3.0 Inter...     514      NaN   
5         Attribution-NonCommercial-ShareAlike 3.0 Inter...    1151      NaN   
10        Attribution-NonCommercial-NoDerivatives (aka M...   50135      NaN   
20        Attribution-NonCommercial-NoDerivatives (aka M...     361      NaN   

                                                 
         number publisher tags            title  
track_id                                         
2             3       NaN   []             Food  
3             4       NaN   []     Electric Ave  
5             6       NaN   []       This World  
10            1       NaN   []          Freeway  
20            3       NaN   []  Spiritual Level  

[5 rows x 52 columns]

In [3]:
mask_small = tracks[('set', 'subset')] == 'small'
tracks_small = tracks[mask_small]

print(tracks_small.shape)
tracks_small[[('set','subset'), ('track','genre_top')]].head()

(8000, 52)


,set,track
,subset,genre_top
track_id,,
2,small,Hip-Hop
5,small,Hip-Hop
10,small,Pop
140,small,Folk
141,small,Folk


In [4]:
mask_labeled = tracks_small[('track', 'genre_top')].notna()

tracks_labeled = tracks_small[mask_labeled]

print(tracks_labeled.shape)


(8000, 52)


In [5]:
track_ids = tracks_labeled.index  
genre_top = tracks_labeled[('track', 'genre_top')]

In [6]:
genrePath = "Data/unzippedRaw/fma_metadata/genres.csv"
genres = pd.read_csv(genrePath, index_col=0)
print(genres.shape)
genres.head()


(163, 4)


,#tracks,parent,title,top_level
genre_id,,,,
1,8693,38,Avant-Garde,38
2,5271,0,International,2
3,1752,0,Blues,3
4,4126,0,Jazz,4
5,4106,0,Classical,5


In [7]:
features_path = "Data/unzippedRaw/fma_metadata/features.csv"

features = pd.read_csv(
    features_path,
    index_col=0,       
    header=[0, 1, 2],  
    engine="python"
)

print(features.shape)
features.head()


(106574, 518)


feature    chroma_cens                                                    \
statistics    kurtosis                                                     
number              01        02        03        04        05        06   
track_id                                                                   
2             7.180653  5.230309  0.249321  1.347620  1.482478  0.531371   
3             1.888963  0.760539  0.345297  2.295201  1.654031  0.067592   
5             0.527563 -0.077654 -0.279610  0.685883  1.937570  0.880839   
10            3.702245 -0.291193  2.196742 -0.234449  1.367364  0.998411   
20           -0.193837 -0.198527  0.201546  0.258556  0.775204  0.084794   

feature                                             ...   tonnetz            \
statistics                                          ...       std             
number            07        08        09        10  ...        04        05   
track_id                                            ...                       
2           1.481593  2.691455  0.866868  1.341231  ...  0.054125  0.012226   
3           1.366848  1.054094  0.108103  0.619185  ...  0.063831  0.014212   
5          -0.923192 -0.927232  0.666617  1.038546  ...  0.040730  0.012691   
10          1.770694  1.604566  0.521217  1.982386  ...  0.074358  0.017952   
20         -0.289294 -0.816410  0.043851 -0.804761  ...  0.095003  0.022492   

feature                     zcr                                          \
statistics             kurtosis       max      mean    median       min   
number            06         01        01        01        01        01   
track_id                                                                  
2           0.012111   5.758890  0.459473  0.085629  0.071289  0.000000   
3           0.017740   2.824694  0.466309  0.084578  0.063965  0.000000   
5           0.014759   6.808415  0.375000  0.053114  0.041504  0.000000   
10          0.013921  21.434212  0.452148  0.077515  0.071777  0.000000   
20          0.021355  16.669037  0.469727  0.047225  0.040039  0.000977   

feature                         
statistics      skew       std  
number            01        01  
track_id                        
2           2.089872  0.061448  
3           1.716724  0.069330  
5           2.193303  0.044861  
10          3.542325  0.040800  
20          3.189831  0.030993  

[5 rows x 518 columns]

In [8]:
print(features.index.dtype, track_ids.dtype)

int64 int64


In [9]:
features_sel = features.loc[track_ids]

print(features_sel.shape)
features_sel.head()

(8000, 518)


feature    chroma_cens                                                    \
statistics    kurtosis                                                     
number              01        02        03        04        05        06   
track_id                                                                   
2             7.180653  5.230309  0.249321  1.347620  1.482478  0.531371   
5             0.527563 -0.077654 -0.279610  0.685883  1.937570  0.880839   
10            3.702245 -0.291193  2.196742 -0.234449  1.367364  0.998411   
140           0.533579 -0.623885 -1.086205 -1.081079 -0.765151 -0.072282   
141           0.172898 -0.284804 -1.169662 -1.062855 -0.706868 -0.708281   

feature                                             ...   tonnetz            \
statistics                                          ...       std             
number            07        08        09        10  ...        04        05   
track_id                                            ...                       
2           1.481593  2.691455  0.866868  1.341231  ...  0.054125  0.012226   
5          -0.923192 -0.927232  0.666617  1.038546  ...  0.040730  0.012691   
10          1.770694  1.604566  0.521217  1.982386  ...  0.074358  0.017952   
140        -0.882913 -0.582376 -0.884749 -0.645214  ...  0.157683  0.028070   
141        -0.204884  0.023624 -0.642770 -0.786291  ...  0.145994  0.024342   

feature                     zcr                                          \
statistics             kurtosis       max      mean    median       min   
number            06         01        01        01        01        01   
track_id                                                                  
2           0.012111   5.758890  0.459473  0.085629  0.071289  0.000000   
5           0.014759   6.808415  0.375000  0.053114  0.041504  0.000000   
10          0.013921  21.434212  0.452148  0.077515  0.071777  0.000000   
140         0.025946  11.052547  0.379395  0.052379  0.036621  0.001953   
141         0.032111  32.994659  0.415527  0.040267  0.034668  0.002930   

feature                         
statistics      skew       std  
number            01        01  
track_id                        
2           2.089872  0.061448  
5           2.193303  0.044861  
10          3.542325  0.040800  
140         3.143968  0.057712  
141         4.204097  0.028665  

[5 rows x 518 columns]

In [10]:
features_sel.columns = [
    "_".join([str(level) for level in col]).strip()
    for col in features_sel.columns.values
]
features_sel.head()

,chroma_cens_kurtosis_01,chroma_cens_kurtosis_02,chroma_cens_kurtosis_03,chroma_cens_kurtosis_04,chroma_cens_kurtosis_05,chroma_cens_kurtosis_06,chroma_cens_kurtosis_07,chroma_cens_kurtosis_08,chroma_cens_kurtosis_09,chroma_cens_kurtosis_10,...,tonnetz_std_04,tonnetz_std_05,tonnetz_std_06,zcr_kurtosis_01,zcr_max_01,zcr_mean_01,zcr_median_01,zcr_min_01,zcr_skew_01,zcr_std_01
track_id,,,,,,,,,,,,,,,,,,,,,
2,7.180653,5.230309,0.249321,1.347620,1.482478,0.531371,1.481593,2.691455,0.866868,1.341231,...,0.054125,0.012226,0.012111,5.758890,0.459473,0.085629,0.071289,0.000000,2.089872,0.061448
5,0.527563,-0.077654,-0.279610,0.685883,1.937570,0.880839,-0.923192,-0.927232,0.666617,1.038546,...,0.040730,0.012691,0.014759,6.808415,0.375000,0.053114,0.041504,0.000000,2.193303,0.044861
10,3.702245,-0.291193,2.196742,-0.234449,1.367364,0.998411,1.770694,1.604566,0.521217,1.982386,...,0.074358,0.017952,0.013921,21.434212,0.452148,0.077515,0.071777,0.000000,3.542325,0.040800
140,0.533579,-0.623885,-1.086205,-1.081079,-0.765151,-0.072282,-0.882913,-0.582376,-0.884749,-0.645214,...,0.157683,0.028070,0.025946,11.052547,0.379395,0.052379,0.036621,0.001953,3.143968,0.057712
141,0.172898,-0.284804,-1.169662,-1.062855,-0.706868,-0.708281,-0.204884,0.023624,-0.642770,-0.786291,...,0.145994,0.024342,0.032111,32.994659,0.415527,0.040267,0.034668,0.002930,4.204097,0.028665


In [11]:
X = features_sel.to_numpy(dtype=np.float32)
X.shape

(8000, 518)

In [12]:
genres_str = genre_top.astype(str)
unique_genres = sorted(genres_str.unique())
unique_genres

['Electronic',
 'Experimental',
 'Folk',
 'Hip-Hop',
 'Instrumental',
 'International',
 'Pop',
 'Rock']

In [13]:
genre_to_idx = {g: i for i, g in enumerate(unique_genres)}
genre_to_idx

{'Electronic': 0,
 'Experimental': 1,
 'Folk': 2,
 'Hip-Hop': 3,
 'Instrumental': 4,
 'International': 5,
 'Pop': 6,
 'Rock': 7}

In [14]:
y = genres_str.map(genre_to_idx).to_numpy(dtype=np.int64)

print(y.shape)
print(y[:10], genres_str.iloc[:10].tolist())

(8000,)
[3 3 6 2 2 1 7 2 2 2] ['Hip-Hop', 'Hip-Hop', 'Pop', 'Folk', 'Folk', 'Experimental', 'Rock', 'Folk', 'Folk', 'Folk']


In [15]:
num_samples, num_features = X.shape
print("Num samples:", num_samples)
print("Num features:", num_features)
print("Num classes:", len(unique_genres))

vals, counts = np.unique(y, return_counts=True)
for v, c in zip(vals, counts):
    print(v, "->", unique_genres[v], ":", c)

Num samples: 8000
Num features: 518
Num classes: 8
0 -> Electronic : 1000
1 -> Experimental : 1000
2 -> Folk : 1000
3 -> Hip-Hop : 1000
4 -> Instrumental : 1000
5 -> International : 1000
6 -> Pop : 1000
7 -> Rock : 1000


In [16]:
features_sel.describe().T[['mean','std','min','max']].head(20)


,mean,std,min,max
chroma_cens_kurtosis_01,0.111983,1.656377,-1.803165,42.851593
chroma_cens_kurtosis_02,0.007180,1.934319,-1.816620,79.781960
chroma_cens_kurtosis_03,0.200494,6.073813,-1.825738,517.172424
chroma_cens_kurtosis_04,0.005261,1.288859,-1.760243,26.313240
chroma_cens_kurtosis_05,0.067485,1.636802,-1.781202,43.405674
chroma_cens_kurtosis_06,0.077369,1.599574,-1.718097,47.604107
chroma_cens_kurtosis_07,0.021349,2.086226,-1.778957,101.148888
chroma_cens_kurtosis_08,0.139799,2.983964,-1.745497,188.753738
chroma_cens_kurtosis_09,0.012609,1.810619,-1.802878,74.518082
chroma_cens_kurtosis_10,0.091662,1.984203,-1.787692,75.878792


In [17]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled.shape

(8000, 518)

In [18]:
X_scaled.mean(), X_scaled.std()

(np.float32(4.4185683e-11), np.float32(0.99999994))

In [19]:
k = 10
nn = NearestNeighbors(
    n_neighbors=k + 1,
    metric="cosine"
)

nn.fit(X_scaled)

NearestNeighbors(metric='cosine', n_neighbors=11)

In [20]:
neighbors = nn.kneighbors(X_scaled[:5])
neighbors

(array([[5.9604645e-08, 4.7944069e-01, 4.8429418e-01, 5.1835275e-01,
         5.2914357e-01, 5.3407586e-01, 5.3549230e-01, 5.3805548e-01,
         5.3903282e-01, 5.4215980e-01, 5.4639095e-01],
        [5.9604645e-08, 4.9328500e-01, 5.1787263e-01, 5.1907897e-01,
         5.3276962e-01, 5.4320747e-01, 5.4618675e-01, 5.4672867e-01,
         5.4871684e-01, 5.5043477e-01, 5.5494779e-01],
        [0.0000000e+00, 3.3217251e-01, 4.5856774e-01, 4.7418493e-01,
         5.0879109e-01, 5.1585478e-01, 5.1687354e-01, 5.1758772e-01,
         5.2244985e-01, 5.3018504e-01, 5.3047484e-01],
        [0.0000000e+00, 4.5047259e-01, 4.8035926e-01, 4.8240459e-01,
         4.9398935e-01, 5.0785112e-01, 5.3350091e-01, 5.3520083e-01,
         5.3564024e-01, 5.4313791e-01, 5.4400945e-01],
        [0.0000000e+00, 3.5856247e-01, 4.3445367e-01, 4.7482824e-01,
         4.7922510e-01, 4.8226577e-01, 4.8826575e-01, 4.8850274e-01,
         4.9494773e-01, 4.9651098e-01, 4.9830246e-01]], dtype=float32),
 array([[   0,  36

In [21]:
distances, indices = nn.kneighbors(X_scaled)
distances.shape, indices.shape

((8000, 11), (8000, 11))

In [22]:
num_nodes = X_scaled.shape[0]

src = []
tgt = []

for i in range(num_nodes):
    neighbors_i = indices[i, 1:]
    for j in neighbors_i:
        src.append(i)
        tgt.append(j)

edge_index = np.vstack([src, tgt])

edge_index.shape

(2, 80000)

In [23]:
edge_index_sym = np.concatenate(
    [edge_index, edge_index[[1, 0], :]],
    axis=1
)

edge_index_sym = np.unique(edge_index_sym, axis=1)

edge_index_sym.shape


(2, 127949)

In [24]:
num_nodes = len(y)
indices_all = np.arange(num_nodes)

idx_train_full, idx_temp, y_train_full, y_temp = train_test_split(
    indices_all,
    y,
    test_size=0.4,
    stratify=y,
    random_state=42
)

idx_val, idx_test, y_val, y_test = train_test_split(
    idx_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

len(idx_train_full), len(idx_val), len(idx_test)

(4800, 1600, 1600)

In [25]:
num_nodes = len(y)

train_mask = np.zeros(num_nodes, dtype=bool)
val_mask = np.zeros(num_nodes, dtype=bool)
test_mask = np.zeros(num_nodes, dtype=bool)

train_mask[idx_train_full] = True
val_mask[idx_val] = True
test_mask[idx_test] = True

train_mask.sum(), val_mask.sum(), test_mask.sum()


(np.int64(4800), np.int64(1600), np.int64(1600))

In [33]:
tracks_subset = tracks.loc[features.index]


track_ids = tracks_subset.index.values
track_titles = tracks_subset[('track', 'title')].fillna("Unknown").values
track_artists = tracks_subset[('artist', 'name')].fillna("Unknown").values

In [30]:
np.savez(
    "fma_graph_data.npz",
    X_scaled=X_scaled.astype(np.float32),
    y=y,
    edge_index=edge_index_sym,
    idx_train=idx_train_full,
    idx_val=idx_val,
    idx_test=idx_test,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask,
    unique_genres=np.array(unique_genres),
    genre_to_idx_keys=np.array(list(genre_to_idx.keys())),
    genre_to_idx_vals=np.array(list(genre_to_idx.values())),
    track_ids=track_ids,
    track_titles=track_titles,
    track_artists=track_artists,
)


In [32]:
data = np.load("fma_graph_data.npz", allow_pickle=True)
list(data.keys())


['X_scaled',
 'y',
 'edge_index',
 'idx_train',
 'idx_val',
 'idx_test',
 'train_mask',
 'val_mask',
 'test_mask',
 'unique_genres',
 'genre_to_idx_keys',
 'genre_to_idx_vals',
 'track_ids',
 'track_titles',
 'track_artists']